# 01 - Read Raw Data, Crop and Filter

In this notebook we use the **MNE sample dataset** (auditory/visual task, MEG + EEG) to:

1. Load a raw `.fif` file
2. Visualize the raw signals
3. Extract events from the stimulus channel
4. Inspect the recording info and sensor locations
5. Mark bad channels and pick a subset of channels
6. Crop and filter the data, and compare the PSD before/after
7. Save the processed data

> The interactive MNE browser opens in a separate Qt window, so screenshots are stored in `saved_figures/`.

## 0. Setup
Import the libraries and choose the matplotlib backend (`QtAgg` = interactive windows).

In [2]:
import matplotlib
import pathlib
import mne

In [ ]:
matplotlib.use('QtAgg')

# use %matplotlib inline in the notebook


In [ ]:
# mne.set_config("MNE_BROWSER_THEME",  "dark") # accept "light", "dark" and "auto"

Download (first time only) and locate the MNE sample dataset.

In [4]:
sample_data_dir = mne.datasets.sample.data_path()

# Convert to a pathlib.Path for more convenience
sample_data_dir = pathlib.Path(sample_data_dir)
sample_data_dir

c:\Users\Zakaria\Downloads\eeg_study\eeg-signal-processing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WindowsPath('C:/Users/Zakaria/mne_data/MNE-sample-data')

## 1. Load the raw data
`mne.io.read_raw` reads the file lazily (data is not loaded into memory yet).

In [5]:
raw_path = sample_data_dir / 'MEG' /'sample'/'sample_audvis_raw.fif'
raw = mne.io.read_raw(raw_path)
raw

Opening raw data file C:\Users\Zakaria\mne_data\MNE-sample-data\MEG\sample\sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.


<Raw | sample_audvis_raw.fif, 376 x 166800 (277.7 s), ~3.2 MiB, data not loaded>

## 2. Visualize the raw data
`raw.plot()` opens the interactive browser where you can scroll through time and channels.

In [ ]:
raw.plot()

Using qt as 2D backend.


Channels marked as bad:
['MEG 2443', 'EEG 053', np.str_('MEG 0233')]
Waiting for Loading-Thread to finish... (max. 10 sec)


In [5]:
%matplotlib inline

# raw.plot()

## 3. Extract events from the STIM channel
Each event is a row `[sample, previous_value, event_code]`.

In [9]:
event = mne.find_events(raw)
# event

Finding events on: STI 014
320 events found on stim channel STI 014
Event IDs: [ 1  2  3  4  5 32]


Map each event code to a readable condition name:

In [10]:
event_id = {
    'Auditory/Left': 1,
    'Auditory/Right':2,
    'Visual/Left': 3,
    'Visual/Right': 4,
    'Smiley': 5,
    'Button' : 32
}
event_id

{'Auditory/Left': 1,
 'Auditory/Right': 2,
 'Visual/Left': 3,
 'Visual/Right': 4,
 'Smiley': 5,
 'Button': 32}

Total number of events:

In [11]:
len(event)

320

Button-press events (code `32`) and how many there are:

In [13]:
event[event[:,2] == 32],len(event[event[:,2] == 32])

(array([[ 34649,      0,     32],
        [ 42938,      0,     32],
        [ 51803,      0,     32],
        [ 59677,      0,     32],
        [ 67782,      0,     32],
        [ 75624,      0,     32],
        [ 85111,      0,     32],
        [ 93018,      0,     32],
        [101540,      0,     32],
        [109324,      0,     32],
        [118369,      0,     32],
        [126470,      0,     32],
        [134287,      0,     32],
        [142815,      0,     32],
        [152336,      0,     32],
        [168672,      0,     32]]),
 16)

Count the visual events (left + right):

In [89]:
left_visual = event[event[:,2] == event_id['Visual/Left']]
right_visual = event[event[:,2] == event_id['Visual/Right']]
visual_event_lenght = len(left_visual) + len(right_visual)
visual_event_lenght

144

In [91]:
left_visual.shape, right_visual.shape

((73, 3), (71, 3))

Plot the raw data with the events shown as vertical lines:

In [90]:
raw.plot(events=event,event_id=event_id)

Channels marked as bad:
['MEG 2443', 'EEG 053', np.str_('MEG 0813'), np.str_('MEG 0523')]


## 4. Gather info about the data
`raw.info` holds the metadata: sampling rate, channel names/types, bad channels, measurement date...

In [92]:
raw.info

<Info | 21 non-empty values
 acq_pars: ACQch001 110113 ACQch002 110112 ACQch003 110111 ACQch004 110122 ...
 bads: 4 items (MEG 2443, EEG 053, MEG 0813, MEG 0523)
 ch_names: MEG 0113, MEG 0112, MEG 0111, MEG 0122, MEG 0123, MEG 0121, MEG ...
 chs: 204 Gradiometers, 102 Magnetometers, 9 Stimulus, 60 EEG, 1 EOG
 custom_ref_applied: False
 description: acquisition (megacq) VectorView system at NMR-MGH
 dev_head_t: MEG device -> head transform
 dig: 146 items (3 Cardinal, 4 HPI, 61 EEG, 78 Extra)
 events: 1 item (list)
 experimenter: MEG
 file_id: 4 items (dict)
 highpass: 0.1 Hz
 hpi_meas: 1 item (list)
 hpi_results: 1 item (list)
 lowpass: 172.2 Hz
 meas_date: 2002-12-03 19:01:10 UTC
 meas_id: 4 items (dict)
 nchan: 376
 proj_id: 1
 proj_name: test
 projs: PCA-v1: off, PCA-v2: off, PCA-v3: off
 sfreq: 600.6 Hz
>

In [93]:
raw.info['meas_date']

datetime.datetime(2002, 12, 3, 19, 1, 10, 720100, tzinfo=datetime.timezone.utc)

In [94]:
raw.info['bads']

['MEG 2443', 'EEG 053', np.str_('MEG 0813'), np.str_('MEG 0523')]

In [95]:
raw.ch_names, len(raw.ch_names)

(['MEG 0113',
  'MEG 0112',
  'MEG 0111',
  'MEG 0122',
  'MEG 0123',
  'MEG 0121',
  'MEG 0132',
  'MEG 0133',
  'MEG 0131',
  'MEG 0143',
  'MEG 0142',
  'MEG 0141',
  'MEG 0213',
  'MEG 0212',
  'MEG 0211',
  'MEG 0222',
  'MEG 0223',
  'MEG 0221',
  'MEG 0232',
  'MEG 0233',
  'MEG 0231',
  'MEG 0243',
  'MEG 0242',
  'MEG 0241',
  'MEG 0313',
  'MEG 0312',
  'MEG 0311',
  'MEG 0322',
  'MEG 0323',
  'MEG 0321',
  'MEG 0333',
  'MEG 0332',
  'MEG 0331',
  'MEG 0343',
  'MEG 0342',
  'MEG 0341',
  'MEG 0413',
  'MEG 0412',
  'MEG 0411',
  'MEG 0422',
  'MEG 0423',
  'MEG 0421',
  'MEG 0432',
  'MEG 0433',
  'MEG 0431',
  'MEG 0443',
  'MEG 0442',
  'MEG 0441',
  'MEG 0513',
  'MEG 0512',
  'MEG 0511',
  'MEG 0523',
  'MEG 0522',
  'MEG 0521',
  'MEG 0532',
  'MEG 0533',
  'MEG 0531',
  'MEG 0542',
  'MEG 0543',
  'MEG 0541',
  'MEG 0613',
  'MEG 0612',
  'MEG 0611',
  'MEG 0622',
  'MEG 0623',
  'MEG 0621',
  'MEG 0633',
  'MEG 0632',
  'MEG 0631',
  'MEG 0642',
  'MEG 0643',
  'MEG

In [96]:
raw.ch_names[320]

'EEG 006'

In [97]:
raw.info['sfreq']

600.614990234375

In [98]:
raw.info['chs'][320]

{'scanno': 321,
 'logno': 6,
 'kind': 2 (FIFFV_EEG_CH),
 'range': 0.00030517578125,
 'cal': 0.00019799999427050352,
 'coil_type': 1 (FIFFV_COIL_EEG),
 'loc': array([ 0.02533827,  0.08910906,  0.09980437,  0.00235201,  0.11096951,
        -0.03500458,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ,  1.        ]),
 'unit': 107 (FIFF_UNIT_V),
 'unit_mul': 0 (FIFF_UNITM_NONE),
 'ch_name': 'EEG 006',
 'coord_frame': 4 (FIFFV_COORD_HEAD)}

## 5. Visualize sensor locations
2D and 3D views of the EEG electrode positions (bad channels are shown in red).

In [99]:
raw.plot_sensors(ch_type='eeg')

<Figure size 800x800 with 1 Axes>

In [100]:
raw.plot_sensors(kind='3d', ch_type = 'eeg')

<Figure size 800x800 with 1 Axes>

## 6. Mark channels as bad
Add `EEG 051` to the list of bad channels and check it on the sensor plot.

In [47]:
raw.info['bads']

['MEG 2443', 'EEG 053', np.str_('MEG 0813'), np.str_('MEG 0523')]

In [101]:
# add bad channels

raw.info['bads'] += ['EEG 051']
raw.plot_sensors(ch_type='eeg')

<Figure size 800x800 with 1 Axes>

## 7. Select a subset of channels
`pick()` keeps only the channel types we want. `exclude=[]` keeps the bad channels too (they stay marked as bad).

### EEG + EOG

In [102]:
raw.info

<Info | 21 non-empty values
 acq_pars: ACQch001 110113 ACQch002 110112 ACQch003 110111 ACQch004 110122 ...
 bads: 5 items (MEG 2443, EEG 053, MEG 0813, MEG 0523, EEG 051)
 ch_names: MEG 0113, MEG 0112, MEG 0111, MEG 0122, MEG 0123, MEG 0121, MEG ...
 chs: 204 Gradiometers, 102 Magnetometers, 9 Stimulus, 60 EEG, 1 EOG
 custom_ref_applied: False
 description: acquisition (megacq) VectorView system at NMR-MGH
 dev_head_t: MEG device -> head transform
 dig: 146 items (3 Cardinal, 4 HPI, 61 EEG, 78 Extra)
 events: 1 item (list)
 experimenter: MEG
 file_id: 4 items (dict)
 highpass: 0.1 Hz
 hpi_meas: 1 item (list)
 hpi_results: 1 item (list)
 lowpass: 172.2 Hz
 meas_date: 2002-12-03 19:01:10 UTC
 meas_id: 4 items (dict)
 nchan: 376
 proj_id: 1
 proj_name: test
 projs: PCA-v1: off, PCA-v2: off, PCA-v3: off
 sfreq: 600.6 Hz
>

In [103]:
# select a subset of data

#exclude mark as give us all the channels knowing that there are bad
raw_eeg = raw.copy().pick(["eeg","eog"], exclude=[])
len(raw_eeg.ch_names)

61

In [104]:
raw_eeg.info

<Info | 20 non-empty values
 acq_pars: ACQch001 110113 ACQch002 110112 ACQch003 110111 ACQch004 110122 ...
 bads: 2 items (EEG 053, EEG 051)
 ch_names: EEG 001, EEG 002, EEG 003, EEG 004, EEG 005, EEG 006, EEG 007, ...
 chs: 60 EEG, 1 EOG
 custom_ref_applied: False
 description: acquisition (megacq) VectorView system at NMR-MGH
 dev_head_t: MEG device -> head transform
 dig: 146 items (3 Cardinal, 4 HPI, 61 EEG, 78 Extra)
 events: 1 item (list)
 experimenter: MEG
 file_id: 4 items (dict)
 highpass: 0.1 Hz
 hpi_meas: 1 item (list)
 hpi_results: 1 item (list)
 lowpass: 172.2 Hz
 meas_date: 2002-12-03 19:01:10 UTC
 meas_id: 4 items (dict)
 nchan: 61
 proj_id: 1
 proj_name: test
 projs: []
 sfreq: 600.6 Hz
>

In [62]:
raw_eeg.plot(events=event, event_id=event_id)

Channels marked as bad:
['EEG 053', 'EEG 051', 'EEG 051', np.str_('EEG 050')]


### MEG only

In [63]:
raw_meg = raw.copy().pick(["meg"],exclude=[])
len(raw_meg)

166800

In [64]:
raw_meg.plot(events=event, event_id=event_id)

Channels marked as bad:
['MEG 2443', np.str_('MEG 0813'), np.str_('MEG 0523')]


## 8. Crop and filter the data

### Crop
Keep only the first 100 seconds of the EEG data.

In [66]:
raw_eeg_cropped = raw_eeg.copy().crop(tmax = 100)
raw_eeg_cropped.times[-1]

np.float64(99.99916914588277)

In [67]:
len(raw_eeg_cropped)

60062

In [68]:
raw_eeg_cropped.plot(events=event, event_id = event_id)

Channels marked as bad:
['EEG 053', 'EEG 051', 'EEG 051', np.str_('EEG 050')]


### Band-pass filter 0.4 - 70 Hz
- **High-pass (`l_freq`)** removes slow drifts.
- **Low-pass (`h_freq`)** removes high-frequency noise.

In [75]:
# high pass filter with cut off 0.4 Hz and low pass filter with cut off 70 Hz
raw_eeg_cropped_filtered = (
    raw_eeg_cropped.copy().load_data().filter(l_freq=0.4,h_freq=70.0)
    )

Reading 0 ... 60061  =      0.000 ...    99.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.4 - 70 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.40
- Lower transition bandwidth: 0.40 Hz (-6 dB cutoff frequency: 0.20 Hz)
- Upper passband edge: 70.00 Hz
- Upper transition bandwidth: 17.50 Hz (-6 dB cutoff frequency: 78.75 Hz)
- Filter length: 4957 samples (8.253 s)



In [ ]:
raw_eeg_cropped.plot(events=event,event_id=event_id,title="unfiltered")
raw_eeg_cropped_filtered.plot(events=event,event_id=event_id,title="filtered_l_0.4_h_70")


Channels marked as bad:
['EEG 053', 'EEG 051', 'EEG 051', np.str_('EEG 050'), np.str_('EEG 014')]
Channels marked as bad:
['EEG 053', 'EEG 051', 'EEG 051', np.str_('EEG 050'), np.str_('EEG 014')]


Unfiltered (left) vs filtered 0.4 - 70 Hz (right):

![unfiltered vs filtered 0.4-70 Hz](saved_figures/raw_eeg_l_0.4_h_70.png)

### Power Spectral Density (PSD) before vs after filtering
The PSD shows the power at each frequency: after filtering, the power above the low-pass cut-off drops sharply.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2)

raw_eeg_cropped.plot_psd(ax=ax[0], show = False)
raw_eeg_cropped_filtered.plot_psd(ax=ax[1], show = False)


ax[0].set_title("psd before filtering")
ax[1].set_title("psd after filtering_l_0.4_h_70")
fig.set_tight_layout(True)
plt.show()


NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Effective window size : 3.410 (s)
Plotting power spectral density (dB=True).
NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Effective window size : 3.410 (s)
Plotting power spectral density (dB=True).


C:\Users\Zakaria\AppData\Local\Temp\ipykernel_4260\989948124.py:12: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.show()
c:\Users\Zakaria\Downloads\eeg_study\eeg-signal-processing\.venv\Lib\site-packages\mne\viz\utils.py:793: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._draw()
c:\Users\Zakaria\Downloads\eeg_study\eeg-signal-processing\.venv\Lib\site-packages\mne\viz\evoked.py:117: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  evoked_fig.canvas.draw()
c:\Users\Zakaria\Downloads\eeg_study\eeg-signal-processing\.venv\Lib\site-packages\mne\viz\evoked.py:178: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  evoked_fig.canvas.draw()
c:\Users\Zakaria\Downloads\eeg_study\eeg-signal-processing\.venv\Lib\site-package

For comparison, an earlier run with a **0.1 - 40 Hz** filter:

![raw EEG filtered 0.1-40 Hz](saved_figures/raw_eeg_l_0.1_h_40.png)

![PSD 0.1-40 Hz](saved_figures/psd_l_0.1_h_40.png)

Clicking a frequency range on the PSD plot shows its topography (here the average over ~130 - 185 Hz):

![PSD topomap](saved_figures/Figure_2.png)

### Band-pass filter 1 - 30 Hz
A narrower band, common for ERP analysis.

In [81]:
raw_eeg_cropped_filtered_2 = raw_eeg_cropped.copy().load_data().filter(
    l_freq = 1.0,
    h_freq = 30.0
    )
raw_eeg_cropped_filtered_2.plot(events = event, event_id=event_id)

Reading 0 ... 60061  =      0.000 ...    99.999 secs...


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1983 samples (3.302 s)



Channels marked as bad:
['EEG 053', 'EEG 051', 'EEG 051', np.str_('EEG 050'), np.str_('EEG 014')]


In [82]:
fig, ax = plt.subplots(2)

raw_eeg_cropped.plot_psd(ax=ax[0], show = False)
raw_eeg_cropped_filtered_2.plot_psd(ax=ax[1], show = False)


ax[0].set_title("psd before filtering")
ax[1].set_title("psd after filtering_l_1.0_h_30")
fig.set_tight_layout(True)
plt.show()

NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Effective window size : 3.410 (s)
Plotting power spectral density (dB=True).
NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Effective window size : 3.410 (s)
Plotting power spectral density (dB=True).


C:\Users\Zakaria\AppData\Local\Temp\ipykernel_4260\3479744057.py:10: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.show()
c:\Users\Zakaria\Downloads\eeg_study\eeg-signal-processing\.venv\Lib\site-packages\mne\viz\utils.py:793: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._draw()
c:\Users\Zakaria\Downloads\eeg_study\eeg-signal-processing\.venv\Lib\site-packages\mne\viz\evoked.py:117: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  evoked_fig.canvas.draw()
c:\Users\Zakaria\Downloads\eeg_study\eeg-signal-processing\.venv\Lib\site-packages\mne\viz\evoked.py:178: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  evoked_fig.canvas.draw()
c:\Users\Zakaria\Downloads\eeg_study\eeg-signal-processing\.venv\Lib\site-packag

![PSD before vs after 1-30 Hz](saved_figures/psd_l_1.0_h_70.png)

## 9. Save the data
Save the cropped + filtered (1 - 30 Hz) EEG data to `out_data/` to use it in the next notebooks.

In [107]:
raw_eeg_cropped_filtered_2.save(pathlib.Path('./out_data')/'eeg_cropped_fit_raw.fif',
                                overwrite=True)

Writing c:\Users\Zakaria\Downloads\eeg_study\eeg-signal-processing\M_EEG_analysis_with_mne_Python\out_data\eeg_cropped_fit_raw.fif


Closing c:\Users\Zakaria\Downloads\eeg_study\eeg-signal-processing\M_EEG_analysis_with_mne_Python\out_data\eeg_cropped_fit_raw.fif
[done]


[WindowsPath('c:/Users/Zakaria/Downloads/eeg_study/eeg-signal-processing/M_EEG_analysis_with_mne_Python/out_data/eeg_cropped_fit_raw.fif')]